# LangChain 核心模块 Agent - Self ask with search

- Google 搜索对接
    - 第三方平台：https://serpapi.com
    - LangChain API 封装：`SerpAPIWrapper`
- LangChain Agent 模块使用
    - Agent 类型：`SELF_ASK_WITH_SEARCH`
    - Agent 实例化：`initialize_agent` 方法
 
LangChain 内置 Agent类型详见：https://python.langchain.com/docs/modules/agents/agent_types/

In [1]:
%%capture --no-stderr
%pip install -U langchain

以下`SERPAPI_API_KEY`仅为示例，请访问 https://serpapi.com 注册账号并替换为自己的 `API_KEY`（每月100次免费调用）

```python
import os

# 更换为自己的 Serp API KEY
os.environ["SERPAPI_API_KEY"] = "xxx"
```

建议将其设置为环境变量，而不是在代码中显式明文设置

In [ ]:
from langchain_openai import OpenAI
from langchain_community.utilities import SerpAPIWrapper
from langchain.agents import AgentExecutor, Tool
from langchain import hub

llm = OpenAI(model_name="gpt-3.5-turbo-instruct", temperature=0)

In [3]:
# 实例化查询工具
search = SerpAPIWrapper()
tools = [
    Tool(
        name="Intermediate Answer",
        func=search.run,
        description="useful for when you need to ask with search",
    )
]

In [ ]:
# 获取 self-ask-with-search 提示词模板并创建 Agent
from langchain.agents import create_self_ask_with_search_agent

prompt = hub.pull("hwchase17/self-ask-with-search")
agent = create_self_ask_with_search_agent(llm, tools, prompt)
self_ask_with_search = AgentExecutor(agent=agent, tools=tools, verbose=True)

In [ ]:
# 实际运行 Agent，查询问题（正确）
self_ask_with_search.invoke(
    {"input": "成都举办的大运会是第几届大运会？2023年大运会举办地在哪里？"}
)

In [ ]:
# 实际运行 Agent，查询问题
self_ask_with_search.invoke(
    {"input": "2023年大运会举办地在哪里？成都举办的大运会是第几届大运会？"}
)

In [8]:
# Reason-only 正确：启发式 Prompt（猜测是大运会新闻报道数据给到了 gpt-3.5-turbo-instruct 模型）
print(llm.invoke("成都举办的大运会是第几届大运会？"))



成都举办的大运会是第31届大运会。


In [9]:
# Reason-only 错误：非启发式 Prompt（容易出现事实类错误，未结合 Web Search 工具）
print(llm.invoke("2023年大运会举办地在哪里？"))



2023年大运会的举办地将在中国的重庆市。


#### 使用 GPT-4 作为大语言模型实现更优的 ReAct 范式

In [10]:
from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [ ]:
agent = create_self_ask_with_search_agent(chat_model, tools, prompt)
self_ask_with_search_chat = AgentExecutor(agent=agent, tools=tools, verbose=True)

In [ ]:
# GPT-4 based ReAct 答案（正确）
self_ask_with_search_chat.invoke(
     {"input": "成都举办的大运会是第几届大运会？2023年大运会举办地在哪里？"}
)